In [ ]:
# Estrutura de tabela de dispersão

guardaRoupas = {
    "divisao_1": "blusa",
    "divisao_2" : "suéter",
    "divisao_3" : "calça"
}

print(guardaRoupas)

exemploHash = hash("divisao_1")
print(exemploHash)

{'divisao_1': 'blusa', 'divisao_2': 'suéter', 'divisao_3': 'calça'}
8911013876926221972


In [2]:
# Coleta de dados meteorológicos

import csv
import requests

URL = "https://api.thingspeak.com/channels/12397/feeds.json"

LIMITE = {"end": "2026-08-01 23:59:59", "results":8000}

resposta = requests.get(URL, params=LIMITE,timeout = 15)
resposta.raise_for_status()
dadosBrutos = resposta.json()

print(dadosBrutos)

informacoesDaEstacao = dadosBrutos["channel"]
leiturasDaEstacao = dadosBrutos["feeds"]

print("=== Informações de coleta ===")
print(f"Dados coletados: {len(leiturasDaEstacao)} da estação {informacoesDaEstacao.get('name')}")
print ("===        ===")

# Criando o dataset

caminhoDoArquivo = "dadosColetados.csv"

tabelaHash = {}
i = 0
for leitura in leiturasDaEstacao:
  entryID = leitura.get("entry_id")

  leituraBruta = 0.0
  chuva = 0.0

  if leitura.get("field5") is not None:
    leituraBruta = float(leitura.get("field5"))

  if (leituraBruta > 0.0):
    chuva = leituraBruta   

  tabelaHash[entryID] = {
      "id": i,
      "criado_em": leitura.get("created_at"),
      "direcao_vento": leitura.get("field1"),
      "velocidade_vento": leitura.get("field2"),
      "umidade": leitura.get("field3"),
      "temperatura": leitura.get("field4"),
      "intensidade_de_chuva": leitura.get("field5"),
      "pressao": leitura.get("field6"),
      "choveu": bool(chuva is not None and chuva > 0.0)
  }

  i+=1

print(tabelaHash)

campos = list(tabelaHash.values())[0].keys()
print(f"campos: {campos}")

with open(caminhoDoArquivo, "w", encoding="utf-8", newline="") as f:
  escritor = csv.DictWriter(f, fieldnames=campos)
  escritor.writeheader()
  for entryID, leitura in tabelaHash.items():
    escritor.writerow(leitura)

{'channel': {'id': 12397, 'name': 'WeatherStation', 'description': 'MathWorks Weather Station, West Garage, Natick, MA 01760, USA', 'latitude': '42.299676', 'longitude': '-71.350525', 'field1': 'Wind Direction (North = 0 degrees)', 'field2': 'Wind Speed (mph)', 'field3': '% Humidity', 'field4': 'Temperature (F)', 'field5': 'Rain (Inches/minute)', 'field6': 'Pressure ("Hg)', 'field7': 'Power Level (V)', 'field8': 'Light Intensity', 'created_at': '2014-05-20T21:50:32Z', 'updated_at': '2025-10-02T21:56:57Z', 'elevation': '60', 'last_entry_id': 5741349}, 'feeds': [{'created_at': '2026-07-27T03:58:12Z', 'entry_id': 5678731, 'field1': '0', 'field2': '0.9', 'field3': '0', 'field4': '0.1', 'field5': '0', 'field6': '29.66', 'field7': '0', 'field8': '0'}, {'created_at': '2026-07-27T03:59:12Z', 'entry_id': 5678732, 'field1': '0', 'field2': '0.9', 'field3': '0', 'field4': '0.1', 'field5': '0', 'field6': '29.66', 'field7': '0', 'field8': '0'}, {'created_at': '2026-07-27T04:00:12Z', 'entry_id': 5678

# Treinamento do Modelo 1 - Sem Balanceamento

In [10]:
import pandas as pd
import numpy as np

# Carregando técnicas de IA/ML
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# Carregando ferramentas de normalização
from sklearn.preprocessing import StandardScaler

# Carregando as métricas
from sklearn.metrics import average_precision_score, precision_recall_curve, f1_score, precision_score, recall_score

# Ferramentas de balanceamento/equilibrio do dataset
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import SMOTE
from imblearn.combine import SMOTEENN

# validação inicial, sem o balanceamento
caminhoCSV = "dadosColetados.csv"

df = pd.read_csv(caminhoCSV)
# display(df)

colunaAlvo = "choveu"
df[colunaAlvo] = df[colunaAlvo].astype(str).str.lower().map({"true":1, "false":0, "1": 1, "0":0})
df = df.sort_values("id").reset_index(drop=True)

print(f"Total de registros: {len(df)}")
print(f"Registros de chuva: {df[colunaAlvo].mean():.2%}")


colunasBase = ["direcao_vento", "velocidade_vento", "umidade", "temperatura", "pressao"]

for coluna in colunasBase:
    df[coluna] = pd.to_numeric(df[coluna], errors="coerce" )

df.dropna(subset=colunasBase)

# Validar a taxa de variação instantanea
for coluna in colunasBase:
    df[f"delta_{coluna}"] = df[coluna].diff().fillna(0)

for coluna in colunasBase:
    df[f"media_{coluna}"] = df[coluna].rolling(window=3).mean()

print(f"Total de colunas {len(df.columns)}")

# Preparando o ambiente e conjunto de dados
normalizador = StandardScaler()

knn = KNeighborsClassifier()
X = df[colunasBase]
Y = df[colunaAlvo]

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

pipelineManual = ImbPipeline([("normalizador", normalizador), ("modelo", knn)])

probKnn = cross_val_predict(estimator=pipelineManual, X=X, y=Y, cv=skf, method="predict_proba")[:,1]

predicoesKnn = (probKnn > 0.5).astype(int)

apKnn = average_precision_score(Y, probKnn)

f1Knn = f1_score(Y, predicoesKnn)

precisao = precision_score(Y, predicoesKnn) if predicoesKnn.sum() > 0 else 0.0

recall = recall_score(Y, predicoesKnn)

resultadoKnnManual = {"nome": "Knn sem balanceamento",
                      "modelo": "Knn",
                      "pr_auc": apKnn,
                      "f1": f1Knn,
                      "precisao": precisao,
                      "recall": recall,
                      "threshold": 0.5}

print(resultadoKnnManual)

!pip install nbimporter
#!jupyter nbconvert --to python "/content/drive/MyDrive/bibrain.ipynb" --output bibrain

from google.colab import drive
import sys
import nbimporter
import os

drive.mount('/content/drive/')
sys.path.append("\\content\\")
print(os.listdir('/content/drive/MyDrive'))

import bibrain as br


# estrategiasBiblioteca = [("Random Forest com peso de classe", "rf", None),
#                          ("KNN com SMOTE", "knn", "smote")]

# resultados = []

# for nome, modelo, tecnica in estrategiasBiblioteca:
#    resultados.append(br.avaliarEstrategia(nome, modelo, tecnica, X, Y))

# print(resultados)


Total de registros: 8000
Registros de chuva: 1.07%
Total de colunas 19
{'nome': 'Knn sem balanceamento', 'modelo': 'Knn', 'pr_auc': np.float64(0.04042309725158562), 'f1': 0.0, 'precisao': 0.0, 'recall': 0.0, 'threshold': 0.5}
Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).
['Classroom', 'Controles', 'Colab Notebooks', 'BRANCH VP', 'Epson iPrint', 'Simple Pie Chart', 'Single Bar Chart', 'Historico Escolar Graduação Edwillie.pdf', 'Diploma Graduação Edwillie .pdf', 'CLIPS - Guia de Usuário.gdoc', 'Atividade 2 - Base de dados (1).gdoc', 'Documento sem título.gdoc', 'Classificação de Revistas e Fichamento.gsheet', 'ex4_3004.clp', 'ex8_3004.clp', 'ex5_3004.clp', 'ex7_3004.clp', 'ex3_3004.clp', 'ex2_3004.clp', 'ex6_3004.clp', 'ex1_3004.clp', 'Edwillie Cardoso - Modelo-Projeto-Pesquisa - SENAI (1).gdoc', 'Sistema-de-Priorizacao-de-Bugs-em-Software.pptx', 'Edwillie Cardoso - Atividade 1 - Definição do 

ModuleNotFoundError: No module named 'bibrain'